# Notebook 02: Statistical Assumptions and Diagnostics for ANCOVA
## Study: NCAL-2026-1259 (Computer Assisted Language Learning)

---

### Key Assumptions Evaluated:
1. **Normality of Residuals**: Shapiro-Wilk test, D'Agostino-Pearson omnibus test, Skewness & Kurtosis.
2. **Homogeneity of Variances**: Levene's test (based on median and mean) across groups for both pre- and post-tests.
3. **Homogeneity of Regression Slopes**: Testing the Group × Pre-test interaction term to ensure ANCOVA slope parallelism.
4. **Linearity**: Correlation and linear regression between pre-test (covariate) and post-test (dependent variable).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATA_CANDIDATES = [
    "data/raw/Hypothetical_EFL_AI_Study_Data.csv",
    "data/Hypothetical_EFL_AI_Study_Data.csv",
    "Hypothetical_EFL_AI_Study_Data.csv",
    "/mnt/data/Hypothetical_EFL_AI_Study_Data.csv",
    "../Hypothetical_EFL_AI_Study_Data.csv"
]

def load_data():
    for p in DATA_CANDIDATES:
        if os.path.exists(p):
            df = pd.read_csv(p)
            df['Group'] = df['Group'].astype(str).str.strip().str.capitalize()
            return df
    raise FileNotFoundError("Data file not found.")

df = load_data()
print(f"Loaded {len(df)} records.")

### Step 1: Normality Diagnostics (Shapiro-Wilk Test)

In [ ]:
normality_results = []
metrics = [
    ('Writing_Pre', 'Writing Post'),
    ('Writing_Post', 'Writing Post'),
    ('SRL_Pre', 'SRL Pre'),
    ('SRL_Post', 'SRL Post'),
    ('AI_Pre', 'AI Pre'),
    ('AI_Post', 'AI Post')
]

for col in ['Writing_Pre', 'Writing_Post', 'SRL_Pre', 'SRL_Post', 'AI_Pre', 'AI_Post']:
    for grp in ['Experimental', 'Control']:
        data = df[df['Group'] == grp][col].dropna()
        w_stat, p_val = stats.shapiro(data)
        normality_results.append({
            'Variable': col,
            'Group': grp,
            'N': len(data),
            'Shapiro_W': w_stat,
            'p_value': p_val,
            'Normal_p05': p_val > 0.05
        })

norm_df = pd.DataFrame(normality_results)
norm_df

### Step 2: Homogeneity of Variance (Levene's Test)

In [ ]:
levene_results = []
for col in ['Writing_Pre', 'Writing_Post', 'SRL_Pre', 'SRL_Post', 'AI_Pre', 'AI_Post']:
    exp_data = df[df['Group'] == 'Experimental'][col].dropna()
    ctrl_data = df[df['Group'] == 'Control'][col].dropna()
    stat_med, p_med = stats.levene(exp_data, ctrl_data, center='median')
    stat_mean, p_mean = stats.levene(exp_data, ctrl_data, center='mean')
    levene_results.append({
        'Variable': col,
        'Levene_Stat_Median': stat_med,
        'p_value_Median': p_med,
        'Levene_Stat_Mean': stat_mean,
        'p_value_Mean': p_mean,
        'Equal_Variance_p05': p_med > 0.05
    })

levene_df = pd.DataFrame(levene_results)
levene_df

### Step 3: Homogeneity of Regression Slopes (Group × Covariate Interaction)

In [ ]:
models = [
    ('Writing Quality', 'Writing_Post', 'Writing_Pre'),
    ('Self-Regulated Learning (SRL)', 'SRL_Post', 'SRL_Pre'),
    ('AI Literacy', 'AI_Post', 'AI_Pre')
]

slope_tests = []
for name, post, pre in models:
    formula = f"{post} ~ {pre} * C(Group, Treatment(reference='Control'))"
    ols_res = smf.ols(formula, data=df).fit()
    interaction_term = [param for param in ols_res.params.index if ':' in param][0]
    p_inter = ols_res.pvalues[interaction_term]
    f_inter = ols_res.tvalues[interaction_term]**2
    slope_tests.append({
        'Construct': name,
        'Interaction_Term': interaction_term,
        'F_Interaction': f_inter,
        'p_Interaction': p_inter,
        'Homogeneity_Holds_p05': p_inter > 0.05
    })

slope_df = pd.DataFrame(slope_tests)
slope_df

### Step 4: Export Diagnostic Summary

In [ ]:
os.makedirs("outputs", exist_ok=True)
norm_df.to_csv("outputs/Assumption_Normality_Shapiro.csv", index=False)
levene_df.to_csv("outputs/Assumption_Levene_Variance.csv", index=False)
slope_df.to_csv("outputs/Assumption_Homogeneity_Slopes.csv", index=False)
print("[SUCCESS] All assumption test results saved to outputs/ directory.")